# IS2083 Lab 5: AI Automation Tool, Part 1

**Before you begin, you need TWO things:**

1. Your **Jupyter login** (you used it to open this notebook).
2. Your **API key** for the class Qwen Coder model server (from your instructor).

This lab assumes you already know basic Python. The new material is the **AI model**, its **API**, and **prompt design**. You will connect to the class **Qwen Coder** model, use it as a coding assistant, and build an **AI-powered security log assessment** tool. At the end you will feed it a real security log exported from **Windows** or **macOS** and have it write an assessment report. Run cells with **Shift + Enter**, top to bottom.

## Step 1: Install the openai client (run once)

In [1]:
%pip install openai

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## Step 2: Connect to the Qwen Coder model server

Paste the `BASE_URL` (ends in `/v1`) and your `API_KEY` from your instructor. The code then queries the server for the served model id.

In [2]:
from openai import OpenAI

BASE_URL = "http://gpustack.cds.utsa.edu/v1"  # class model server address
API_KEY  = "gpustack_2dd9292f3707c9ac_04518bf39ae93a5cd2942c10cb71c754"  # your personal API key

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
MODEL = client.models.list().data[1].id   # discover the served model id
print("Connected. Model:", MODEL)

Connected. Model: qwen3.5-122b-a10b


## Step 3: A reusable model helper (with retries)

`ask_model` wraps one chat completion and retries on transient errors.

In [4]:
import time

def ask_model(prompt: str, temperature: float = 1.0, retries: int = 2) -> str:
    """Send one prompt to Qwen Coder and return its text reply, retrying on failure."""
    for attempt in range(retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                top_p=0.95,
            )
            return resp.choices[0].message.content
        except Exception:
            if attempt == retries:
                raise
            time.sleep(1.5 * (attempt + 1))

print(ask_model("Say hello in one short sentence."))



Hello there.


## Step 4: Plan the tool

Input: one security log entry. Output: a structured result with a `label` (BENIGN / SUSPICIOUS / MALICIOUS) and a one-sentence `reason`. You solve the structure problem with a strict output format in the prompt plus defensive parsing.

## Step 5: Use Qwen Coder as a coding assistant

Ask the model to sketch an approach, then keep only what you understand.

In [5]:
question = (
    "I am writing a Python function that asks an LLM to classify a security log entry "
    "and returns a dataclass with label and reason. Suggest a robust prompt format and "
    "a defensive way to parse the reply. Keep it concise."
)
print(ask_model(question))



```python
import json
import re
from dataclasses import dataclass

@dataclass
class LogClassification:
    label: str
    reason: str

def classify_log(log_entry: str, llm_call) -> LogClassification:
    ALLOWED_LABELS = {"SAFE", "MALICIOUS", "SUSPICIOUS"}
    
    # 1. Robust Prompt (JSON-only constraint)
    prompt = f"""
    Role: Security Analyst. Task: Classify log.
    Allowed Labels: {ALLOWED_LABELS}.
    Output: Valid JSON ONLY. Format: {{"label": "...", "reason": "..."}}.
    Constraints: No markdown, no conversational text.
    Log: {log_entry}
    """
    
    # 2. Defensive Parsing
    try:
        # Mock API call (use temp=0.0 in production)
        raw_response = llm_call(prompt) 
        
        # Extract JSON block ignoring markdown fences or chatter
        match = re.search(r'\{[\s\S]*\}', raw_response, re.DOTALL)
        content = match.group(0) if match else raw_response
        
        data = json.loads(content)
        
        # Validate schema and allowed va

## Step 6: Define the result type and parsers

An `Enum` fixes the labels, a `dataclass` gives a typed result, and two regexes extract the fields.

In [7]:
from dataclasses import dataclass
from enum import Enum
import re

class Label(str, Enum):
    BENIGN = "BENIGN"
    SUSPICIOUS = "SUSPICIOUS"
    MALICIOUS = "MALICIOUS"
    UNKNOWN = "UNKNOWN"

@dataclass
class Triage:
    line: str
    label: Label
    reason: str

LABEL_RE = re.compile(r"^LABEL:\s*(\w+)", re.IGNORECASE | re.MULTILINE)
REASON_RE = re.compile(r"^REASON:\s*(.+)", re.IGNORECASE | re.MULTILINE)

## Step 7: Name your tool

This tool can go in your professional portfolio, so give it a name you would be proud to show. `triage_log` is a fine name for the function, but the **tool** itself can be called anything: Play it straight (SentinelScan, ThreatLens, LogHawk, WatchTower) or give it some personality (SnitchBot, LogFather, GrepReaper, Sherlock Logs, The Log Whisperer, LogNessMonster), your call. Set it here; it appears in your report and your repository name.

In [8]:
TOOL_NAME = "LogSentry"   # <-- change this to your own creative name

## Step 8: Build the tool

Write the classifier: a strict prompt, one `ask_model` call, then defensive parsing that falls back to `Label.UNKNOWN`. (You may rename `triage_log` to match your tool if you like.)

In [10]:
def triage_log(line: str) -> Triage:
    prompt = (
        "You are a security log triage assistance.\n"
        "Classify the log line as BENIGN, SUSPICIOUS, or MALICIOUS "
        "and give a one sentence reason.\n"
        "Reply in EXACTLY this format:\n"
        "LABEL: <BENIGN|SUSPICIOUS|MALICIOUS>\n"
        "REASON: <one sentence>\n\n"
        f"Log line:\n{line}\n"
    )

    text = ask_model(prompt)

    label_match = LABEL_RE.search(text)
    raw = label_match.group(1).upper() if label_match else "UNKNOWN"

    label = Label(raw) if raw in Label.__members__ else Label.UNKNOWN

    reason_match = REASON_RE.search(text)
    reason = reason_match.group(1).strip() if reason_match else text.strip()

    return Triage(line=line, label=label, reason=reason)

## Step 9: Quick sanity check
Confirm the tool works on one entry before feeding it a file.

In [11]:
print(triage_log("Failed password for root from 203.0.113.9 port 4022 ssh2"))

Triage(line='Failed password for root from 203.0.113.9 port 4022 ssh2', label=<Label.SUSPICIOUS: 'SUSPICIOUS'>, reason='This event indicates an unauthorized authentication attempt against the root account, which is a common indicator of brute-force or scanning activity.')


## Step 10: Assess a real security log file

Export your computer's security log to **one file** and upload it here. Do the ONE that matches your computer; you only turn in one (Windows or macOS).

**Windows (Event Viewer):** open Event Viewer, expand **Windows Logs**, click **Security**. In the right panel click **Save All Events As...**, set *Save as type* to **CSV (Comma Separated)**, and save it as `security_log.csv`.

**macOS (Terminal):** export recent security-related events to one file:

```
log show --predicate 'process == "sudo" OR process == "sshd" OR process == "loginwindow" OR eventMessage CONTAINS[c] "authentication"' --last 6h --style syslog > security_log.txt
```

If that file is nearly empty, use: `log show --last 1h --style syslog > security_log.txt`

Then click **Upload** in the Jupyter file browser and add your `security_log.csv` (Windows) or `security_log.txt` (macOS) to this notebook's folder. Set `LOG_FILE` below to match.

In [12]:
LOG_FILE = "security_log.txt"   # Windows CSV, or "security_log.txt" on macOS
MAX_ENTRIES = 20                 # keep this first version small and quick

def read_log_lines(path, limit=MAX_ENTRIES):
    with open(path, encoding="utf-8", errors="ignore") as f:
        lines = [ln.strip() for ln in f if ln.strip()]
    return lines[:limit]

entries = read_log_lines(LOG_FILE)
results = [triage_log(line) for line in entries]
print(f"Assessed {len(results)} entries from {LOG_FILE}")

Assessed 20 entries from security_log.txt


## Step 11: Write the assessment report

Have your tool write a Markdown report named after your tool. This report is your deliverable.

In [13]:
from datetime import datetime

def write_report(results, tool_name, path="assessment.md"):
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"# {tool_name} Security Log Assessment\n\n")
        f.write(f"Generated: {datetime.now():%Y-%m-%d %H:%M}\n\n")
        f.write(f"Entries assessed: {len(results)}\n\n")
        for r in results:
            f.write(f"- **{r.label.value}** {r.reason}\n")
            f.write(f"  - `{r.line}`\n")
    return path

report_path = write_report(results, TOOL_NAME)
print("Wrote", report_path, "\n")
print(open(report_path, encoding="utf-8").read())

Wrote assessment.md 

# LogSentry Security Log Assessment

Generated: 2026-11-10 17:10

Entries assessed: 20

- **BENIGN** The line appears to be a log format header or template containing placeholders rather than actual system activity.
  - `Timestamp                       (process)[PID]`
- **BENIGN** This log entry is a standard informational message from the trusted Apple coreauthd daemon indicating an internal state change within the LocalAuthentication framework.
  - `2026-11-10 11:05:03.973386-0600  localhost coreauthd[751]: (LocalAuthenticationCore) [com.apple.LocalAuthentication:Notifications] Will post com.apple.LocalAuthentication.environment.StateDidChange`
- **BENIGN** This log entry reflects normal macOS system behavior recording a physical lid closure event configured to keep the device awake.
  - `2026-11-10 11:05:03.985932-0600  localhost loginwindow[468]: [com.apple.loginwindow.logging:Standard] clamshellStateChanged | Clamshell state changed: closed=1, shouldSleepWhen

## Step 12: Read, test, and trust

Double-click to edit and answer briefly:

1. Why model the output with an `Enum` and `dataclass` instead of a bare dict?
2. What does your defensive parsing do when the model ignores the format?
3. Which entries in your report look wrong, and how would you catch that?

_Your answers:_
1. An Enum makes sure that the classifications are using consistent and predefined values. A data class makes the ouput a lot easier to orgnize and understand by keeping the log entry, label, and reason together.
2. Defensive parsing makes sure that the program doesnt crash when the AI gives an unexpected response. If the label is missing or its not valid, the program just gives it unknown and uses the available text as the reason.
3. They all look right to me, you would catch it by reading through the sample by hand and double checking if the reasoning makes sense to you.

## Step 13: Save and submit

1. **File > Save Notebook**, then **File > Download** for the `.ipynb`.
2. Download your report `assessment.md` too.
3. On GitHub, create a **public** repository **named after your tool** (for example `sentinelscan`). Do **not** name it lab5 or use the course name.
4. **Add file > Upload files**: upload your notebook and `assessment.md`. Do **not** upload the raw security log; it may contain private data.
5. Before uploading, **remove your real API key** from the connect cell.
6. Take the screenshots listed in the handout into the submission template.